In [ ]:
import pandas as pd
import re
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel

In [ ]:
#model path
mpath = "/workspace/Llama-2-7b-chat-hf"
#gpu
# device_map = {'':1}
device_map={"": 0}
#dataset path
dataset_test = "test_ds.csv"
#new model path
fn_model = "Llama-2-7b-ovi-fn"

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    mpath,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map=device_map,
)
model = PeftModel.from_pretrained(base_model, fn_model)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(mpath, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:

df = pd.read_csv(dataset_test)

RE_EMOJI = re.compile('[\U00010000-\U0010ffff]', flags=re.UNICODE)

def clean_text(text):
    # Remove non-ASCII characters
    cleaned_text = re.sub(r'[^\x00-\x7FЋ]+', '', text)
    # Remove other unwanted characters or patterns
    cleaned_text = re.sub(r'[\[\]]', '', cleaned_text)  # Remove square brackets
    # Add more patterns to remove as needed
    cleaned_text = RE_EMOJI.sub(r'', cleaned_text)
    
    # Split the text into paragraphs based on newline characters
    paragraphs = cleaned_text.split('\n')
    
    # Initialize variables to store the longest paragraph and its length
    longest_paragraph = ''
    max_length = 0
    
    # Iterate through each paragraph
    for paragraph in paragraphs:
        # Split each paragraph by the specified sequence of characters ('   ')
        sub_paragraphs = paragraph.split('   ')
        
        # Iterate through each sub-paragraph
        for sub_paragraph in sub_paragraphs:
            # Calculate the length of the current sub-paragraph
            length = len(sub_paragraph)
            
            # Update the longest paragraph if the current one is longer
            if length > max_length:
                max_length = length
                longest_paragraph = sub_paragraph
    
    return longest_paragraph

def generate_response(prompt):
    pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=600, return_full_text=False)
    result = pipe(f"<s>[INST] {prompt}[/INST]")[0]['generated_text']
    return clean_text(result)


i = 0
for index, row in df.iterrows():
    print(f"Processing row {i}")
    df.at[index, 'Generated'] = generate_response(row['Input'])
    i += 1

df.to_csv("new_ds_test.csv", index=False)